In [ ]:
# 1. Uninstall the conflicting version
!pip uninstall timm -y

# 2. Install the stable version for ImageBind
!pip install timm==0.6.13

# 3. Patch the code to ensure imports match this version
# This fixes the specific error you saw by pointing it to the correct location in timm 0.6.x
!sed -i "s/from timm.layers import trunc_normal_/from timm.models.layers import trunc_normal_/g" /content/ImageBind/imagebind/models/multimodal_preprocessors.py
!sed -i "s/from timm.layers import/from timm.models.layers import/g" /content/ImageBind/imagebind/models/imagebind_model.py

print("Environment Fixed.")

Found existing installation: timm 0.6.13
Uninstalling timm-0.6.13:
  Successfully uninstalled timm-0.6.13
  Using cached timm-0.6.13-py3-none-any.whl.metadata (38 kB)
Using cached timm-0.6.13-py3-none-any.whl (549 kB)


Environment Fixed.


In [ ]:
import os
import requests
import time

# Create directory
os.makedirs(".data/audio", exist_ok=True)

# Define robust URLs from Wikimedia Commons (WAV/OGG formats supported by torchaudio)
audio_urls = {
    # Class 1: Animals
    "animal_dog": "https://upload.wikimedia.org/wikipedia/commons/c/ce/Sound_of_dog.ogg",
    "animal_cow": "https://upload.wikimedia.org/wikipedia/commons/a/aa/Cow_moo_1.ogg",
    "animal_rooster": "https://upload.wikimedia.org/wikipedia/commons/7/7b/Rooster_crowing.ogg",

    # Class 2: Transport/City
    "transport_train": "https://upload.wikimedia.org/wikipedia/commons/1/12/Train_horn_low.ogg",
    "transport_siren": "https://upload.wikimedia.org/wikipedia/commons/5/52/Siren_noise.ogg",
    "transport_car": "https://upload.wikimedia.org/wikipedia/commons/5/5a/Traffic_noise_01.ogg",

    # Class 3: Music/Instruments
    "music_piano": "https://upload.wikimedia.org/wikipedia/commons/c/c4/Piano_Scale_C_Major.ogg",
    "music_guitar": "https://upload.wikimedia.org/wikipedia/commons/3/36/A_acoustic_guitar_chord.ogg",
    "music_drum": "https://upload.wikimedia.org/wikipedia/commons/1/12/Drum_roll_sound_effect.ogg"
}

headers = {'User-Agent': 'Mozilla/5.0'}
audio_paths = []
labels_true = []

print("Downloading Audio Samples...")
for name, url in audio_urls.items():
    file_ext = url.split('.')[-1]
    save_path = f".data/audio/{name}.{file_ext}"

    try:
        r = requests.get(url, headers=headers)
        with open(save_path, 'wb') as f:
            f.write(r.content)

        audio_paths.append(save_path)
        labels_true.append(name.split('_')[0]) # e.g., 'animal'
        print(f" - Downloaded: {name}")
        time.sleep(0.5) # Be nice to the server
    except Exception as e:
        print(f"Error downloading {name}: {e}")

print(f"\nPrepared {len(audio_paths)} audio files.")

 - Downloaded: animal_dog
 - Downloaded: animal_cow
 - Downloaded: animal_rooster
 - Downloaded: transport_train
 - Downloaded: transport_siren
 - Downloaded: transport_car
 - Downloaded: music_piano
 - Downloaded: music_guitar
 - Downloaded: music_drum

Prepared 9 audio files.


In [ ]:
import sys
import os
import torch

# 1. Patch the failing file (transformer.py)
# This changes 'from timm.layers' -> 'from timm.models.layers'
!sed -i "s/from timm.layers import/from timm.models.layers import/g" /content/ImageBind/imagebind/models/transformer.py

# 2. Re-apply previous patches (just in case)
!sed -i "s/from timm.layers import/from timm.models.layers import/g" /content/ImageBind/imagebind/models/multimodal_preprocessors.py
!sed -i "s/from timm.layers import/from timm.models.layers import/g" /content/ImageBind/imagebind/models/imagebind_model.py

# 3. Ensure Path
if '/content/ImageBind' not in sys.path:
    sys.path.append('/content/ImageBind')

# 4. Import
from imagebind import data
from imagebind.models import imagebind_model
from imagebind.models.imagebind_model import ModalityType

# 5. Load Model
print("Loading ImageBind Model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = imagebind_model.imagebind_huge(pretrained=True)
model.eval()
model.to(device)

print("\nSUCCESS: Model loaded!")

Loading ImageBind Model...


100%|██████████| 4.47G/4.47G [01:41<00:00, 47.4MB/s]



SUCCESS: Model loaded!


In [ ]:
# 1. Install standard audio backend
!pip install soundfile

# 2. Re-import torchaudio to register the backend
import torchaudio
import torch

# 3. Check if 'audio_paths' exists (in case you restarted and lost variables)
# If this prints an error, scroll up and run the "Data Acquisition" cell again!
try:
    if 'audio_paths' not in globals() or len(audio_paths) == 0:
        print("WARNING: 'audio_paths' is empty or missing.")
        print("Please scroll up and re-run the 'Data Acquisition' cell (Step 3) to download the audio files.")
    else:
        print(f"Found {len(audio_paths)} audio paths ready for processing.")

        # 4. Run the Processing Code
        from imagebind import data
        from imagebind.models.imagebind_model import ModalityType

        # Ensure model is on the correct device (it should be from previous step)
        device = "cuda" if torch.cuda.is_available() else "cpu"

        print("Encoding Audio...")

        # Load audio using the default backend (now usually 'soundfile')
        inputs = {
            ModalityType.AUDIO: data.load_and_transform_audio_data(audio_paths, device)
        }

        with torch.no_grad():
            embeddings_dict = model(inputs)

        embeddings = embeddings_dict[ModalityType.AUDIO].cpu().numpy()
        print(f"SUCCESS: Audio Embeddings Shape: {embeddings.shape}")

except Exception as e:
    print(f"\nAn error occurred: {e}")
    print("If this is a backend error, try restarting the runtime one last time and running the cells in order.")

Found 9 audio paths ready for processing.
Encoding Audio...

An error occurred: TorchCodec is required for load_with_torchcodec. Please install torchcodec to use this function.
If this is a backend error, try restarting the runtime one last time and running the cells in order.


In [ ]:
import torch
from imagebind import data
from imagebind.models import imagebind_model
from imagebind.models.imagebind_model import ModalityType

# 1. Load Model
print("Loading ImageBind Model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = imagebind_model.imagebind_huge(pretrained=True)
model.eval()
model.to(device)

# 2. Process Audio
# ImageBind handles resampling and spectrogram conversion internally
inputs = {
    ModalityType.AUDIO: data.load_and_transform_audio_data(audio_paths, device)
}

# 3. Generate Embeddings
print("Encoding Audio...")
with torch.no_grad():
    embeddings_dict = model(inputs)

embeddings = embeddings_dict[ModalityType.AUDIO].cpu().numpy()
print(f"Audio Embeddings Shape: {embeddings.shape}")

Loading ImageBind Model...


ImportError: TorchCodec is required for load_with_torchcodec. Please install torchcodec to use this function.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
import pandas as pd

# 1. Perform Clustering
k = 3
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
labels_pred = kmeans.fit_predict(embeddings)

# 2. Calculate Metrics
# Silhouette: How similar is an object to its own cluster compared to other clusters? (-1 to 1)
sil_score = silhouette_score(embeddings, labels_pred)

# ARI: Similarity between true labels and predicted labels (0 = random, 1 = perfect)
ari_score = adjusted_rand_score(labels_true, labels_pred)

# NMI: Normalized Mutual Information (0 to 1)
nmi_score = normalized_mutual_info_score(labels_true, labels_pred)

print("--- Clustering Quality Measures ---")
results_df = pd.DataFrame({
    'Metric': ['Silhouette Score (Internal)', 'Adjusted Rand Index (External)', 'Normalized Mutual Info (External)'],
    'Value': [sil_score, ari_score, nmi_score],
    'Interpretation': [
        'Higher is better (Closeness of clusters)',
        '1.0 = Perfect match with True Labels',
        '1.0 = Perfect correlation'
    ]
})
print(results_df)

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Reduce Dimensions
pca = PCA(n_components=2)
reduced_embeddings = pca.fit_transform(embeddings)

# 2. Plotting
plt.figure(figsize=(12, 8))

# Define markers/colors
unique_true = list(set(labels_true))
markers = ['o', 's', '^']

for i, category in enumerate(unique_true):
    # Get indices for this true category
    indices = [idx for idx, label in enumerate(labels_true) if label == category]
    points = reduced_embeddings[indices]

    plt.scatter(points[:, 0], points[:, 1],
                s=200,
                marker=markers[i],
                label=f"True Class: {category}",
                edgecolors='black',
                alpha=0.8)

    # Annotate with specific sound name (e.g., "dog", "siren")
    for j, idx in enumerate(indices):
        fname = os.path.basename(audio_paths[idx]).split('.')[0]
        # Clean up name for display
        short_name = fname.split('_')[-1]
        plt.text(points[j, 0], points[j, 1]+0.015, short_name, fontsize=10, ha='center')

plt.title(f"Audio Clustering with ImageBind\nSilhouette: {sil_score:.2f} | ARI: {ari_score:.2f}")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()